In [ ]:
# impoting the necessary libraries
import pandas as pd

# STEP 1: LOAD RAW DATASETS




In [ ]:
player_info_df = pd.read_csv('afl_players_info_raw.csv')
player_seasonal_stats_df = pd.read_csv('afl_players_seasonal_stats_raw.csv')

/tmp/ipykernel_2893/564511939.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  player_seasonal_stats_df = pd.read_csv('/content/afl_players_seasonal_stats_raw.csv')


In [131]:
print("--- Initial Shapes ---")
print(f"Player Info shape: {player_info_df.shape}")
print(f"Seasonal Stats shape: {player_seasonal_stats_df.shape}\n")

--- Initial Shapes ---
Player Info shape: (2848, 16)
Seasonal Stats shape: (25491, 54)



# STEP 2: CLEANING PLAYER INFO DATASET

Rationale: Drop non-essential URL/text columns that have massive missing volumes
and do not affect performance analysis or record merging.

In [132]:
player_info_df = player_info_df.drop(columns=['profile_pic', 'player_common_names'])

Rationale: Fill minor missing teams (94 rows) with 'Unknown' to preserve baseline
demographic counts.

In [133]:
player_info_df['player_teams'] = player_info_df['player_teams'].fillna('Unknown')

 Rationale: Convert raw date strings into standard datetime objects to allow
 future temporal filtering, career length calculations, and age-based validation.

In [134]:
date_columns = ['born_date', 'debut_date', 'last_date']
for col in date_columns:
    player_info_df[col] = pd.to_datetime(player_info_df[col], errors='coerce')

Rationale: Remove identical duplicate rows to prevent distorted entity counting
and false keys during dataset joining.

In [135]:
player_info_df = player_info_df.drop_duplicates()

# STEP 3: CLEANING SEASONAL STATS DATASET


> Add blockquote



 Rationale: In sports performance metrics, missing numeric data points (such as
goals, tackles, or votes) represent zero recorded instances during those game blocks.

In [136]:
player_seasonal_stats_df = player_seasonal_stats_df.fillna(0.0)

Rationale: Remove duplicated statistical records to protect aggregate performance
metrics from inflation.

In [137]:
player_seasonal_stats_df = player_seasonal_stats_df.drop_duplicates()

In [138]:
player_seasonal_stats_df.shape
player_info_df.shape

(2843, 14)

# STEP 4: DATA TYPE HARMONIZATION & MERGING



Aligning key data types: player_info_df['id'] is int64, while
player_seasonal_stats_df['player_id'] is an object type.

In [141]:
player_seasonal_stats_df['player_id']

,player_id
0,43261
1,43261
2,43261
3,43261
4,43262
...,...
25486,ID_43738
25487,ID_43660
25488,ID_46153
25489,ID_44965


In [142]:
# Harmonize player_id: Remove 'ID_' prefix if present and convert to int64
player_seasonal_stats_df['player_id'] = (
    player_seasonal_stats_df['player_id']
    .astype(str)
    .str.replace('ID_', '', regex=False)
)
player_seasonal_stats_df['player_id'] = player_seasonal_stats_df[
    'player_id'
].astype('int64')

print(
    "Player ID data types successfully harmonized. Seasonal stats player_id"
    f" dtype: {player_seasonal_stats_df['player_id'].dtype}"
)

Player ID data types successfully harmonized. Seasonal stats player_id dtype: int64


Rationale: Perform an inner join on player identifiers to synthesize baseline
attributes with comprehensive temporal performance records.

In [143]:
merged_players_df = pd.merge(
    player_seasonal_stats_df,
    player_info_df,
    left_on='player_id',
    right_on='id',
    how='inner'
)

# STEP 5: POST-MERGE VALIDATION

In [144]:
info_ids = set(player_info_df['id'])
stats_ids = set(player_seasonal_stats_df['player_id'])

unmatched_in_stats = stats_ids - info_ids
unmatched_in_info = info_ids - stats_ids

print("--- Validation Report Summary ---")
print(f"Final Player Info Shape: {player_info_df.shape}")
print(f"Final Seasonal Stats Shape: {player_seasonal_stats_df.shape}")
print(f"Final Merged Dataset Shape: {merged_players_df.shape}")
print(f"Orphan Seasonal Stats (Unmatched IDs): {len(unmatched_in_stats)}")
print(f"Profiles with No Seasonal Stats: {len(unmatched_in_info)}")
print(f"Total Null Values Post-Cleaning: {merged_players_df.isnull().sum().sum()}")

--- Validation Report Summary ---
Final Player Info Shape: (2843, 14)
Final Seasonal Stats Shape: (25481, 54)
Final Merged Dataset Shape: (25081, 68)
Orphan Seasonal Stats (Unmatched IDs): 266
Profiles with No Seasonal Stats: 0
Total Null Values Post-Cleaning: 0




# STEP 6: EXPORT

In [145]:

player_info_df.to_csv('players_info.csv', index=False)
player_seasonal_stats_df.to_csv('seasonal_stats.csv', index=False)
merged_players_df.to_csv('merged_players.csv', index=False)

print("\nAll cleaned files successfully exported: 'players_info.csv', 'seasonal_stats.csv', 'merged_players.csv'")


All cleaned files successfully exported: 'players_info.csv', 'seasonal_stats.csv', 'merged_players.csv'
